# EDA - Machine Maintenance Data

I used this notebook to understand the machine readings before our group trains the models. I checked missing values, unusual sensor readings and which columns might give away the maintenance label. The original CSV is not changed.

## 1. Load the dataset


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
paths = [Path("data/proj_machine_readings.csv"), Path("../data/proj_machine_readings.csv"), Path("proj_machine_readings.csv")]
path = next((p for p in paths if p.exists()), None)
if path is None:
    raise FileNotFoundError("Put proj_machine_readings.csv in the data folder first.")
df = pd.read_csv(path, parse_dates=["timestamp"])
print("Rows and columns:", df.shape)
print("Machines:", df["machine_id"].nunique())
df.head()


## 2. Get a basic idea of the data


In [ ]:
df.info()


In [ ]:
df.describe(include="all").T


**Finding:** There are 100,000 readings from 50 machines, recorded over time. `info()` shows that temperature has fewer non-empty values than the other columns. From `describe()`, vibration ranges from about -107 to 114, humidity stays between 30 and 80, and pressure is between 1 and 5.

These ranges give me a starting point for checking unusual values. They also show that the sensors use different scales.

## 3. Check missing values


In [ ]:
missing = pd.DataFrame({"Missing count": df.isna().sum(), "Missing %": (df.isna().mean() * 100).round(2)})
missing[missing["Missing count"] > 0]


In [ ]:
df.assign(temp_missing=df["temperature"].isna()).groupby("maintenance_required")["temp_missing"].mean().mul(100).round(2)


**Finding:** Temperature has 14% missing values (14,000 rows). The other columns do not have missing values.

Will try median imputation. The median is less affected by extreme readings than the mean. I also want to keep a missing-temperature indicator so the model can test whether missing readings matter. The next cell shows the result using the median from the earlier 80% of rows only.

### Try filling the missing temperature readings

The data is already in time order. I use the earlier 80% of rows to calculate the median, then apply that same value to the later 20%. This is just a simple EDA example; in Week 7 the preprocessing needs to be fitted on the actual training split inside the pipeline.

In [ ]:
time_df = df.sort_values("timestamp").reset_index(drop=True)
split = int(len(time_df) * 0.80)
train_rows = time_df.iloc[:split]
train_median = train_rows["temperature"].median()

eda_filled = time_df.copy()  # Do not change the original df
eda_filled["temperature_was_missing"] = eda_filled["temperature"].isna().astype(int)
eda_filled["temperature_filled"] = eda_filled["temperature"].fillna(train_median)

print("Training rows:", split, "| Later rows:", len(time_df) - split)
print("Training median:", round(train_median, 2))
print("Missing before:", time_df["temperature"].isna().sum())
print("Missing after:", eda_filled["temperature_filled"].isna().sum())
print("Before / after summary:")
print(pd.DataFrame({
    "Before": time_df["temperature"].describe(),
    "After": eda_filled["temperature_filled"].describe()
}).round(2))

plt.figure(figsize=(9, 4))
sns.histplot(time_df["temperature"].dropna(), bins=45, stat="density", color="steelblue", alpha=0.45, label="Before")
sns.histplot(eda_filled["temperature_filled"], bins=45, stat="density", color="orange", alpha=0.35, label="After")
plt.title("Temperature before and after median filling")
plt.xlabel("Temperature")
plt.legend()
plt.tight_layout()
plt.show()


**Finding:** The missing temperatures are filled in the copied dataset, so the missing count becomes zero. There is a taller bar around the median because the missing values are replaced with the same number.


## 4. Check the target

The column we want to predict is `maintenance_required`. 0 means no maintenance and 1 means maintenance is needed. Check how many readings are in each class.

In [ ]:
counts = df["maintenance_required"].value_counts().sort_index()
print(pd.DataFrame({"Count": counts, "Percentage": (counts / len(df) * 100).round(2)}))
plt.bar(counts.index.astype(str), counts.values)
plt.title("Maintenance Required")
plt.xlabel("Maintenance required (0 = No, 1 = Yes)")
plt.ylabel("Number of readings")
plt.show()


**Finding:** About 80% of the readings are class 0 and 20% are class 1. There are a lot more 0s.

**Notes** If the model keeps guessing 0, it can still get high accuracy. I will check precision, recall, F1-score and the confusion matrix when we train the models.

## 5. Look at the sensor values

Use histograms to see the spread of the readings and check for values that look odd.

In [ ]:
sensor_cols = ["temperature", "vibration", "humidity", "pressure", "energy_consumption"]
df[sensor_cols].hist(bins=35, figsize=(12, 7))
plt.suptitle("Distribution of Sensor Readings")
plt.tight_layout()
plt.show()


### Check negative vibration readings

I noticed some vibration values are below zero, so I check how many there are and compare their maintenance rates.

In [ ]:
negative = df["vibration"] < 0
comparison = df.groupby(negative).agg(
    readings=("vibration", "size"),
    avg_size=("vibration", lambda s: s.abs().mean()),
    maintenance_rate=("maintenance_required", "mean")
)
comparison.index = ["Non-negative", "Negative"]
comparison["maintenance_rate"] *= 100
print(comparison.round(2))
print("Negative vibration percentage:", round(negative.mean() * 100, 2), "%")


**Finding:** About 6.7% of vibration readings are negative. The average size and maintenance rate are very similar for negative and non-negative readings. This suggests the sign may not be very useful, but it does not prove there is a sensor error.

**Notes*** Will try absolute vibration as another feature because it shows how strong the vibration is regardless of direction.

In [ ]:
df["vibration_abs"] = df["vibration"].abs()
df[["vibration", "vibration_abs"]].head()


## 6. Check outliers

Use boxplots and the IQR method to find readings that are much higher or lower than most of the data.

In [ ]:
outlier_cols = ["temperature", "vibration_abs", "humidity", "pressure", "energy_consumption"]
fig, axes = plt.subplots(1, 5, figsize=(16, 4))
for ax, col in zip(axes, outlier_cols):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

def count_iqr_outliers(values):
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    return int(((values < q1 - 1.5 * iqr) | (values > q3 + 1.5 * iqr)).sum())

pd.DataFrame({"Outlier count": {c: count_iqr_outliers(df[c].dropna()) for c in outlier_cols}})


**Finding:** The boxplots show some unusual temperature and vibration values. The table above counts outliers using the IQR method. A value outside the usual range is not automatically a wrong reading.

**Notes:** I will keep these values for now because unusually high readings might be signs that a machine needs maintenance.

### Check extreme sensor readings against the anomaly flag

Also check whether the really high temperature and vibration readings appear in the anomaly rows.

In [ ]:
print("Temperature and vibration ranges by anomaly flag:")
print(df.groupby("anomaly_flag")[["temperature", "vibration"]].agg(["min", "max"]).round(2))

extreme_temp = df["temperature"] > 90
extreme_vibration = df["vibration"].abs() > 80
print("\nTemperature > 90 vs anomaly flag:")
print(pd.crosstab(extreme_temp, df["anomaly_flag"]))
print("\nAbsolute vibration > 80 vs anomaly flag:")
print(pd.crosstab(extreme_vibration, df["anomaly_flag"]))


**Finding:** Rows without an anomaly flag have temperature no higher than 90 and vibration between about -80 and 80. Readings above these limits only appear in anomaly-flagged rows.

**Result** These extreme values are linked to the anomaly labels, so removing them could throw away useful information. Not every IQR outlier is an anomaly, though.

## 7. Check which columns relate to maintenance

Use a heatmap and a few graphs to compare the columns with `maintenance_required`.

In [ ]:
corr = df.select_dtypes(include="number").corr()
plt.figure(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()
corr["maintenance_required"].sort_values(ascending=False)


**Finding:** `downtime_risk`, `anomaly_flag` and `machine_status` have strong links with maintenance. Temperature has a moderate positive correlation (about 0.28), while `predicted_remaining_life` has a negative correlation (about -0.34). Humidity, pressure and energy consumption are all close to zero correlation with maintenance.

**Notes** Temperature looks worth testing. The near-zero columns may be less useful on their own, but I will not remove them based only on this graph because Pearson correlation checks linear relationships, not every possible pattern. The strong derived columns need a leakage check.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.boxplot(data=df, x="maintenance_required", y="temperature", ax=axes[0])
axes[0].set_title("Temperature vs Maintenance")
sns.boxplot(data=df, x="maintenance_required", y="vibration_abs", ax=axes[1])
axes[1].set_title("Vibration vs Maintenance")
rates = df.groupby("machine_status")["maintenance_required"].mean()
axes[2].bar(rates.index.astype(str), rates.values)
axes[2].set_title("Maintenance Rate by Status")
axes[2].set_ylabel("Maintenance rate")
plt.tight_layout()
plt.show()


**Finding:** Temperature and vibration look worth testing in the model. I need to be careful with `machine_status` because it can already tell us when a machine has failed.

## 8. Check for data leakage

Data leakage is when the model gets information it would not have at the time of prediction. Check if any columns make the answer too obvious.

In [ ]:
for col in ["anomaly_flag", "machine_status", "failure_type"]:
    print("\n", col)
    print(pd.crosstab(df[col], df["maintenance_required"]))


**Finding:** `anomaly_flag = 1`, `machine_status = 2` and non-Normal `failure_type` are only seen in maintenance class 1. That could make the prediction too easy if these columns are not actually known when a prediction is made.

**Action:** I will check `downtime_risk` and `predicted_remaining_life` below. For the first ML models, I would start with sensor readings and compare any extra features only after confirming they are available at prediction time.

In [ ]:
print("Anomaly flag compared with downtime risk:")
print(pd.crosstab(df["anomaly_flag"], df["downtime_risk"] > 0))
print("\nDowntime risk range for each flag:")
print(df.groupby("anomaly_flag")["downtime_risk"].agg(["min", "max", "mean"]).round(3))

unflagged_maintenance = df[
    (df["maintenance_required"] == 1) &
    (df["anomaly_flag"] == 0) &
    (df["downtime_risk"] == 0)
]
print("\nMaintenance rows with neither flag:", len(unflagged_maintenance))
print("Remaining-life summary for those rows:")
print(unflagged_maintenance["predicted_remaining_life"].describe().round(2))
print("Percent with remaining life <= 24:",
      round(unflagged_maintenance["predicted_remaining_life"].le(24).mean() * 100, 2), "%")
print("\nMaintenance by remaining life <= 24:")
print(pd.crosstab(df["predicted_remaining_life"] <= 24,
                  df["maintenance_required"], normalize="index").mul(100).round(2))


**Finding:** In this dataset, `downtime_risk > 0` matches `anomaly_flag = 1`. There are **10,781** maintenance rows with neither flag. Their median predicted remaining life is **211**, and only around **18%** have 24 or fewer hours remaining. So the claim that nearly all of these readings have 24 hours or less is not supported.

**Next steps:** I will check how `predicted_remaining_life` and `machine_status` were produced before using them in a model. They could give the model information it would not have at prediction time.

## 9. Compare the machines

There are 50 machines, so I check whether some have more maintenance readings than others.

In [ ]:
machine_rates = df.groupby("machine_id")["maintenance_required"].agg(["mean", "count"])
machine_rates["maintenance_percent"] = machine_rates["mean"] * 100
print(machine_rates.sort_values("maintenance_percent", ascending=False).head(10).round(2))
plt.figure(figsize=(12, 4))
plt.bar(machine_rates.index.astype(str), machine_rates["maintenance_percent"])
plt.xticks(rotation=90, fontsize=7)
plt.title("Maintenance Rate by Machine")
plt.xlabel("Machine ID")
plt.ylabel("Maintenance readings (%)")
plt.tight_layout()
plt.show()


**Finding:** Maintenance rates are quite similar across the 50 machines, ranging from about 17.3% to 21.5%. No machine stands out as having a much higher maintenance rate.

`machine_id` does not look very useful by itself for predicting maintenance.

## 10. Compare readings for class 0 and class 1

I compare the average sensor values for readings with and without maintenance.

In [ ]:
summary = df.groupby("maintenance_required")[["temperature", "vibration_abs", "humidity", "pressure", "energy_consumption"]].mean().round(2)
summary


**Finding:** The maintenance class has a higher average temperature (about 80.75 compared with 73.63) and higher absolute vibration (about 53.44 compared with 49.18). Humidity, pressure and energy consumption have almost the same averages in both classes.

**Why it matters:** Temperature and vibration look more promising for predicting maintenance. I will still test the other sensors because averages alone do not show every possible relationship.

## 11. What I found

These are the main points from my EDA:

- There are 100,000 readings from 50 machines. I checked the ranges and found some high temperature and vibration values.
- Temperature has 14% missing values. I tried median filling using the earlier 80% of rows and compared the distribution before and after.
- Around 80% of readings are class 0, so I will use recall and F1-score as well as accuracy.
- About 6.7% of vibration readings are negative. Their size and maintenance rate are similar to the positive readings, so I will test `vibration_abs`.
- Extreme temperature and vibration readings are linked to anomaly rows. I will not remove them without checking.
- Temperature and vibration differ more between maintenance classes than humidity, pressure and energy consumption.
- Some columns may leak the answer. `downtime_risk` matches the anomaly flag, while `machine_status` and `predicted_remaining_life` need checking before use.
- Maintenance rates across machines are similar, so machine ID may not add much on its own.

**Next step:** In Week 7, our group will put the cleaning steps into the ML pipeline, train at least three models and compare their results. This EDA does not change the original CSV.